```{index} string; methods
```

(string-methods)=

# Strings and Their Methods

```{contents}
:local:
:depth: 2
```

Business data is mostly text: customer names, email addresses, product codes, and notes. A C# `string` is an object that comes with many ready-made methods for working with that text. In this section you are the **consumer** of methods someone else wrote. To use one, you need to know only three things: its name, the parameters it expects, and what it returns. How the method does its work is the writer's concern, not yours.[^0504_strings-string-class]

You call a string method with dot notation: the string, a dot, the method name, and the arguments in parentheses, as in `email.Trim()`.

Connect this lesson to writing your own helpers: a caller supplies arguments, receives a typed result, and must obey the contract whether the implementation is yours or the library's. Properties such as Length, indexers such as text[index], instance methods such as text.Trim(), and static calls such as string.IsNullOrWhiteSpace(text) have different syntax.

```text
string result = text.MethodName(arguments);
bool answer = text.TestMethod(arguments);
int position = text.IndexOf(searchText);
char character = text[index];
int size = text.Length;
```

These are patterns, not one program. Choose an actual method and compatible arguments. Keep returned values you need; do not mistake an immutable string operation for an in-place edit. Section 5.3 explains string.Format and returned formatting before the lab uses them.


```{index} string; Length, string; indexing
```



```{index} Length and Indexing
```

## Length and Indexing



`Length` is a property, not a method, so it has no parentheses. It gives the number of UTF-16 code units; for the simple Latin-letter examples here these correspond to the visible characters. Some Unicode symbols occupy more than one code unit. Square brackets with an index give one character (a `char`). Indexes start at 0, so the last character is at `Length - 1`.

In [1]:
using System;

string customer = "Ana Diaz";

Console.WriteLine(customer.Length);              // 8 characters, including the space
Console.WriteLine(customer[0]);                  // A (the first character)
Console.WriteLine(customer[customer.Length - 1]); // z (the last character)

8
A
z


An index outside `0` to `Length - 1` stops the program with an `IndexOutOfRangeException`. Chapter 6 shows how to handle errors like this one.

```{index} string; Trim, string; ToUpper, string; ToLower
```



```{index} Cleaning Text
```

## Cleaning Text



Text typed by people or copied from other systems often has stray spaces and inconsistent capitalization. Clean it before comparing or storing it:

In [1]:
using System;

string rawEmail = "   Ana.Diaz@Example.COM  ";

string trimmed = rawEmail.Trim();     // removes spaces at both ends
string email   = trimmed.ToLower();   // one consistent case for comparisons

Console.WriteLine($"[{rawEmail}]");
Console.WriteLine($"[{email}]");
Console.WriteLine(string.IsNullOrWhiteSpace("   "));  // True: nothing but spaces

[   Ana.Diaz@Example.COM  ]
[ana.diaz@example.com]
True


| Method | Parameters | Returns |
|---|---|---|
| `Trim()` | none | a copy without leading and trailing whitespace |
| `ToUpper()`, `ToLower()` | none | a copy in upper or lower case |
| `string.IsNullOrWhiteSpace(s)` | the string to check | `true` if `s` is null, empty, or only whitespace |

`IsNullOrWhiteSpace` is called on the `string` type itself rather than on one string, because it must also work when there is no string at all (`null`).

```{index} Guided Practice: Normalize a Product Code
```

### Guided Practice: Normalize a Product Code


Codes are typed with stray spaces and in any case. Clean `" sku-1042 "` so the output is `[SKU-1042]`. The starter changes the case but leaves the spaces.


In [ ]:
using System;

string typed = " sku-1042 ";

// Your code starts here.
string code = typed.ToUpper();
// Your code ends here.

Console.WriteLine($"[{code}]");


In [ ]:
using System;

// Solution
string typed = " sku-1042 ";

string code = typed.Trim().ToUpper();

Console.WriteLine($"[{code}]");


The calls are chained: `Trim` returns a new string without the spaces, and `ToUpper` is called on that result. Neither call changes `typed`, which still holds the original text with its spaces.


```{index} string; immutability
```



```{index} Strings Are Immutable
```

## Strings Are Immutable



A string never changes after it is created. Methods such as `ToUpper` do not modify the string they are called on; they return the resulting string value rather than editing the original. If you want to keep the result, assign it to a variable.

In [1]:
using System;

string region = "midwest";
string label  = region.ToUpper();

Console.WriteLine($"region = {region}");  // unchanged
Console.WriteLine($"label  = {label}");   // a new string

region = midwest
label  = MIDWEST


The original region remains midwest; label is MIDWEST. The common mistake `region.ToUpper();` discards the result. It is legal but does not update region. Assign back if that is the intended policy: `region = region.ToUpperInvariant();`. Invariant case conversion is useful for the simple program-owned codes here; names and natural-language text may need a different culture policy.


```{index} string; Contains, string; IndexOf, string; StartsWith
```



```{index} Searching
```

## Searching



These methods answer questions about a string without changing it. `IndexOf` returns the position of the first match, or `-1` when there is no match.

In [1]:
using System;

string sku = "ELEC-1042-BLK";

Console.WriteLine(sku.StartsWith("ELEC"));  // True: an electronics product
Console.WriteLine(sku.Contains("BLK"));     // True: the color code appears
Console.WriteLine(sku.IndexOf('-'));        // 4: position of the first dash
Console.WriteLine(sku.IndexOf("RED"));      // -1: not found

True
True
4
-1


The output is `True`, `True`, `4`, and `-1`. The first two methods answer yes-or-no questions; `IndexOf` answers *where*, and `-1` is its way of saying nowhere. Because `0` is a real position, the common mistake is treating the result as a Boolean: `if (sku.IndexOf("ELEC"))` does not compile, and `if (sku.IndexOf("ELEC") > 0)` wrongly reports a match at the very start as missing. Compare with `>= 0`.


```{index} string; Substring
```



```{index} Extracting Part of a String
```

## Extracting Part of a String



The two forms are:

```csharp
text.Substring(start, length)   // length characters beginning at index start
text.Substring(start)           // from start to the end
```

`Substring(start, length)` returns `length` characters beginning at index `start`. With one argument, `Substring(start)` returns everything from `start` to the end. Combined with `IndexOf`, it can pull the parts out of a structured code such as a SKU (stock keeping unit):

In [1]:
using System;

string sku = "ELEC-1042-BLK";

int dash = sku.IndexOf('-');
string category = sku.Substring(0, dash);         // "ELEC": from index 0, dash characters
string number   = sku.Substring(dash + 1, 4);     // "1042": 4 characters after the dash
string color    = sku.Substring(sku.Length - 3);  // "BLK": from index Length - 3 to the end

Console.WriteLine($"Category: {category}");
Console.WriteLine($"Number:   {number}");
Console.WriteLine($"Color:    {color}");

Category: ELEC
Number:   1042
Color:    BLK


The output is `Category: ELEC`, `Number:   1042`, and `Color:    BLK`. Each extraction is computed from positions, so it depends on the code's format: a SKU with a five-digit number would make `Substring(dash + 1, 4)` return the wrong text, and a start or length that runs past the end throws `ArgumentOutOfRangeException`. The Guard the Format section below checks the format first for that reason.


```{index} string; Replace
```



```{index} Replacing Text
```

## Replacing Text



`Replace(oldText, newText)` returns a copy with every occurrence replaced. For a case-insensitive search, pass `StringComparison.OrdinalIgnoreCase` to `Contains` or `IndexOf`.

In [1]:
using System;

string note = "Ship to: 12 Main St.\tDeliver by Friday";

string cleaned = note.Replace("\t", " ").Replace("St.", "Street");
Console.WriteLine(cleaned);

bool urgent = note.Contains("friday", StringComparison.OrdinalIgnoreCase);
Console.WriteLine($"Mentions Friday: {urgent}");

Ship to: 12 Main Street Deliver by Friday
Mentions Friday: True


The output is `Ship to: 12 Main Street Deliver by Friday` and `Mentions Friday: True`. The two `Replace` calls are chained, each working on the result of the previous one, and `note` itself is unchanged. `Replace` is always case-sensitive, so it would leave `st.` alone; the case-insensitive comparison applies to the search methods, which is why the `Contains` call finds `Friday` from `friday`.


```{index} string; chaining method calls
```



```{index} Chaining Method Calls
```

## Chaining Method Calls



Because each method in this chain returns a string result, you can call the next method directly on that result. Read a chain from left to right:

In [1]:
using System;

string rawCode = "  elec-1042-blk ";

// Each call returns a new string, and the next method is called on that result.
string code = rawCode.Trim().ToUpper();
Console.WriteLine(code);                              // ELEC-1042-BLK
Console.WriteLine(code.Substring(0, code.IndexOf('-')));  // ELEC

ELEC-1042-BLK
ELEC


```{figure} ../../figures/string_method_chain.svg
:alt: A raw SKU is trimmed, then uppercased, with intermediate returned strings and the original raw value unchanged.

A chain passes each returned value to the next operation.
```

The diagram uses ToUpperInvariant for program-owned codes; the earlier ToUpper example uses the current culture. Read left to right: raw input → trimmed result → normalized result. Keeping intermediate variables is often clearer when diagnosing a failed transformation.

| Expression for the supplied example | Result | Meaning |
| --- | --- | --- |
| `rawCode.Trim()` | elec-1042-blk | Strip surrounding whitespace only. |
| The result's `ToUpper()` | ELEC-1042-BLK | Normalize case under the current culture. |
| `code.IndexOf('-')` | 4 | Locate the first separator. |
| `code.Substring(0, 4)` | ELEC | Extract four code units before the separator. |

**Check the precondition:** IndexOf returns -1 if no separator is found. Passing that -1 as a substring length does not mean “not found”; it is an invalid range. An empty string also cannot be indexed at zero. Decide whether malformed input is rejected or given a fallback before extracting parts.


Keep chains short enough to read at a glance. If a chain grows long, store intermediate results in well-named variables.

```{index} Guard the Format Before Extracting
```



```{index} Guard the Format Before Extracting
```

(safe-string-extraction)=

## Guard the Format Before Extracting




```{index} Guard the Format Before Extracting
```


In [1]:
using System;
Console.WriteLine(CategoryOf("  elec-1042-blk "));
Console.WriteLine(CategoryOf("PEN"));
Console.WriteLine(CategoryOf(""));
static string CategoryOf(string raw)
{
    string normalized = raw.Trim().ToUpperInvariant();
    int dash = normalized.IndexOf('-');
    if (dash <= 0) return "Unknown";
    return normalized.Substring(0, dash);
}


ELEC
Unknown
Unknown


This contract accepts a non-null string and returns the nonempty prefix before the first dash, or Unknown when that prefix is unavailable. It does not validate an entire SKU. The results are ELEC, Unknown, and Unknown. A format check is distinct from complete business validation: the number and color segments could still be malformed. The original fixed-format extraction example assumes those segments already meet its stated shape.

Try a leading dash, repeated separators, surrounding whitespace, and lowercase text. Explain which cases this helper accepts and which a complete SKU validator would need to check. Keep validation within current decisions/string skills; regular expressions are not a prerequisite.


```{index} Guided Practice: Clean a Department Code
```



```{index} Guided Practice: Clean a Department Code
```

## Guided Practice: Clean a Department Code




```{index} Guided Practice: Clean a Department Code
```


Accept a non-null raw code, trim surrounding whitespace, and return its invariant uppercase form. If the trimmed code is empty, return Unknown. Print the two results from the caller. Then test a code that is already normalized.


In [ ]:
// Exercise: Guided Practice: Clean a Department Code
// exercise-id: cscs-methods-guided-m5-d6fabf59
using System;
string raw = "  offc  ";
Console.WriteLine(NormalizeDepartment(raw));
Console.WriteLine(NormalizeDepartment("   "));
static string NormalizeDepartment(string value)
{
    // Your code starts here.
    return value; // Replace the placeholder.
    // Your code ends here.
}


In [1]:
// Solution: Guided Practice: Clean a Department Code
using System;
string raw = "  offc  ";
Console.WriteLine(NormalizeDepartment(raw));
Console.WriteLine(NormalizeDepartment("   "));
static string NormalizeDepartment(string value)
{
    string trimmed = value.Trim();
    if (trimmed.Length == 0) return "Unknown";
    return trimmed.ToUpperInvariant();
}


OFFC
Unknown


```{index} string; method summary
```



```{index} Summary
```

## Summary



| Member | Example | Result |
|---|---|---|
| `Length` | `"Ana".Length` | `3` |
| `[index]` | `"Ana"[0]` | `'A'` |
| `Trim()` | `" Ana ".Trim()` | `"Ana"` |
| `ToUpper()` / `ToLower()` | `"Ana".ToUpper()` | `"ANA"` |
| `Contains(text)` | `"ELEC-1042".Contains("10")` | `true` |
| `StartsWith(text)` | `"ELEC-1042".StartsWith("EL")` | `true` |
| `IndexOf(text)` | `"ELEC-1042".IndexOf('-')` | `4` |
| `Substring(start, length)` | `"ELEC-1042".Substring(5, 4)` | `"1042"` |
| `Replace(old, new)` | `"St.".Replace("St.", "Street")` | `"Street"` |

Chapter 9 builds on these methods to split whole lines of text into fields and to validate structured records.

Continue with [String and method practice](assignments/additional_practice.ipynb).

```{rubric} Footnotes
```


[^0504_strings-string-class]: Microsoft, .NET API reference: [String class](https://learn.microsoft.com/en-us/dotnet/api/system.string).